# Adversarial dosage sweep — does retraining ever beat this attacker?

**The question.** Our tabular headline is ASR = 1.000 at every round: three rounds of
adversarial retraining prevented zero evasions. The writeup explains that as a dosage
problem — 400 adversarial rows folded unweighted into 196,001, about 0.2% of the
training mass. That is a *hypothesis*, and shipping a hypothesis in place of a result
is the thing this project spends its methodology section arguing against.

This notebook tests it. Each arm re-runs the loop with adversarial rows carrying
`sample_weight = w`, sweeping `w` wide enough to bracket the transition: at `w=1` they
are 0.2% of training mass, at `w=5000` they outweigh the entire legitimate trainset.

**All three outcomes are publishable**, which is why it is worth the compute:

| Outcome | What we report |
|---|---|
| ASR falls at some dosage | The dosage where the defence starts working, and its PR-AUC cost |
| ASR never falls | Our own "dosage was too small" excuse is refuted by experiment |
| ASR falls only as PR-AUC collapses | The defence is real but not worth buying |

---

## Before you run

1. **Add the data.** *Add Input* → search `kartik2112/fraud-detection` → Add.
2. **Turn Internet ON** (Settings panel) so the repo can be cloned.
3. **Save Version → Save & Run All (Commit)** — that is what runs it overnight,
   detached from your browser. Interactive sessions die when the tab closes.

Expected runtime is roughly 2–4 hours on the full 1.85M rows, inside Kaggle's 12h
commit limit. Results are written after **every arm**, so a run that dies in its last
arm still leaves the arms that finished.


In [ ]:
# --- 1. code ---------------------------------------------------------------------
# No pip install: the sweep pulls only pandas, numpy, sklearn, scipy and xgboost,
# all of which Kaggle preinstalls. Cloning rather than pasting keeps this notebook
# honest -- it runs the same engine as the repo, not a copy that has drifted from it.
!git clone --depth 1 https://github.com/Aditya-Patil27/mastercard-adversarial-payments.git /kaggle/working/repo 2>&1 | tail -2

import sys
sys.path.insert(0, '/kaggle/working/repo/src')

import subprocess
sha = subprocess.run(['git', '-C', '/kaggle/working/repo', 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip()
print('repo at', sha)


In [ ]:
# --- 2. data ---------------------------------------------------------------------
# Kaggle mounts the dataset read-only, where kagglehub cannot reach it and there is no
# network inside the loader. SPARKOV_CSV_DIR points the existing loader at the mount,
# so parsing, feature engineering and the schema contract are the identical code path
# used locally -- not a second loader that could drift.
import os, glob

candidates = glob.glob('/kaggle/input/*/')
print('inputs mounted:', candidates)
root = next((c for c in candidates if 'fraud' in c.lower()), None)
assert root, 'Add the kartik2112/fraud-detection dataset via Add Input'
os.environ['SPARKOV_CSV_DIR'] = root
print('using', root, '->', [p.split('/')[-1] for p in glob.glob(root + '**/*.csv', recursive=True)])


In [ ]:
# --- 3. sanity check before committing hours -------------------------------------
# A 2-minute check that the data parses and the schema contract holds. If this fails,
# it fails now rather than four hours into an unattended run.
from adversarial_payments.data.load import load_features
from adversarial_payments.schema import TARGET

probe = load_features(sample_rows=20_000)
print(f'{len(probe):,} rows, {int(probe[TARGET].sum()):,} fraud '
      f'({probe[TARGET].mean():.4%} base rate)')
assert probe[TARGET].sum() > 0, 'no positives -- wrong dataset or a parsing failure'


In [ ]:
# --- 4. the sweep ----------------------------------------------------------------
# ROWS = 0 uses all 1.85M rows. Lower it to 400_000 if you want a result sooner;
# the arms stay comparable to each other either way, since every arm sees the
# identical split and an identical round 0.
ROWS      = 0
ROUNDS    = 3
ATTEMPTS  = 800
WEIGHTS   = '1 10 50 200 1000 5000'

!cd /kaggle/working/repo && python scripts/run_dosage_sweep.py \
    --rows {ROWS} --rounds {ROUNDS} --attempts {ATTEMPTS} --weights {WEIGHTS} \
    --out /kaggle/working/dosage_sweep.json


In [ ]:
# --- 5. read the result ----------------------------------------------------------
import json, pandas as pd

data = json.load(open('/kaggle/working/dosage_sweep.json'))
rows = [r for arm in data['arms'] for r in arm['rounds']]
df = pd.DataFrame(rows)

print(f"rows={data['rows']:,}  train={data['n_train']:,}  test={data['n_test']:,}")
print(f"attempts/round={data['attempts_per_round']}  fpr_budget={data['fpr_budget']}")
print()
print(df.pivot(index='adversarial_weight', columns='round',
               values=['asr', 'pr_auc']).to_string())


In [ ]:
# --- 6. the answer, stated plainly -----------------------------------------------
final = df[df['round'] == df['round'].max()]
base  = float(df[(df['round'] == 0)]['asr'].iloc[0])
best  = final.loc[final['asr'].idxmin()]

print(f'round-0 ASR (undefended)      : {base:.3f}')
print(f'best final ASR across dosages : {best["asr"]:.3f} at weight {best["adversarial_weight"]:g}')
print(f'  its PR-AUC                  : {best["pr_auc"]:.4f}')
print()
if best['asr'] >= base - 1e-9:
    print('VERDICT: ASR never fell, at any dosage tried -- up to adversarial rows',
          'outweighing the entire legitimate trainset.')
    print('The "dosage was too small" explanation is REFUTED by our own experiment.')
    print('Report: adversarial retraining does not defeat this attacker at any',
          'dosage we can afford, and say so before a judge asks.')
else:
    print('VERDICT: ASR fell. Report the dosage, the ASR, AND the PR-AUC cost together --')
    print('a defence that works by destroying the detector is not a defence.')


## After it finishes

Download `dosage_sweep.json` from the notebook Output, drop it into `artifacts/attack/`,
and commit. Whatever it says goes into the writeup as-is — including, and especially,
the outcome where our own stated explanation turns out to be wrong.
